# What do the models output on unseen instructions? — post-hoc diagnostics for BOTH experiments

This notebook **does not train, fine-tune or re-select anything**. It reads the *saved* test predictions of all
trained models from both experiments (`distinct_pairs`, `all_pairs`) and asks what the models systematically output
on strings they were not trained on (singles, unseen pairs, all 27 triples).

**Task reminder.** 6x6 grids of digits 0-9; `A` = shift right by one with wrap, `B` = rotate 90° clockwise,
`C` = mirror left-right; strings run **left to right** (`AB` = B(A(X))).

**What is measured** (all on frozen predictions):
1. *Audit*: integer exact/cell counts recomputed from raw records; stored targets re-derived by an independent oracle;
   grid/string alignment across models and experiments; free-running vs teacher-forced provenance.
2. *Legal-transformation matching*: every prediction is compared with **all 288** transformations generated by A, B, C
   (36 translations x 8 square orientations). Exact matches can be unique, ambiguous, or absent.
3. *Pre-declared error hypotheses* (identity, first/last op, prefixes/suffixes, dropped/reordered ops, collapsed repeats,
   doubled singles, training-pair transformations).
4. *Stability across grids*: one fixed transformation, a bijective source permutation, and a many-to-one source-copy
   map are **fitted to model predictions on half of the base grids** and evaluated on the other half (DIAG split).
5. *Composition consistency* of the fitted single-operation maps, and the explicit **half-period ambiguity**
   A'=ZA, B'=ZB, C'=ZC (Z = shift by 3 right and 3 down), which leaves every length-2 string unchanged.
6. *Calibration controls* showing why exact accuracy 0 with cell accuracy ~0.10 can still be a perfectly structured error.

**Interpretation limits.** Behavioural compatibility is not a mechanistic or causal claim; DIAG-split fidelities
describe frozen predictions and are not a new benchmark; one training seed only.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION ------------------------------------------------------------------------------------
# Results root used by notebooks 01/02 (their saved outputs show
# /content/drive/MyDrive/decompositionality_results/<experiment>). Change if you used another RESULTS_ROOT.
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
EXPERIMENT_RESULT_DIRS = {
    "distinct_pairs": f"{RESULTS_ROOT}/distinct_pairs",
    "all_pairs": f"{RESULTS_ROOT}/all_pairs",
}
DIAGNOSTICS_OUTPUT_DIR = os.environ.get("DECOMP_DIAGNOSTICS_DIR") or f"{RESULTS_ROOT}/diagnostics_generalization_failures"
MODELS = None                         # None = every model discovered in the result folders
CHUNK_SIZE = 2000                     # records per vectorized matching chunk (memory ~ CHUNK_SIZE x 288 x 36 bytes)
USE_CACHE = True                      # reuse per-run results keyed by prediction-file hash + analysis version
SPLIT_SEED = 20251106                 # deterministic FIT/DIAG split of base-grid IDs (identical for all runs)
HASH_CHECKPOINTS = True               # record SHA-256 of best.pt for provenance (read-only)
RUN_FROZEN_INFERENCE_IF_MISSING = False   # opt-in: regenerate MISSING predictions from best.pt (GPU helpful; never trains)
USE_SYNTHETIC_FIXTURES = os.environ.get("DECOMP_SYNTHETIC_FIXTURES") == "1"  # demo/test ONLY: planted fake models
# --------------------------------------------------------------------------------------------------------------

if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

if USE_SYNTHETIC_FIXTURES:
    import tempfile
    from decompositionality.diagnosis.fixtures import write_fixture_results, PLANTED
    _fx = pathlib.Path(tempfile.mkdtemp()) / "SYNTHETIC_results"
    EXPERIMENT_RESULT_DIRS = write_fixture_results(_fx, n_grids=40)
    DIAGNOSTICS_OUTPUT_DIR = str(_fx / "diagnostics_generalization_failures")
    print("*** SYNTHETIC FIXTURE MODE: planted fake behaviours, NOT real model outputs ***")
    for k, v in PLANTED.items():
        print(f"   {k}: {v}")

from decompositionality.diagnosis.runner import DiagnosticsConfig, run_diagnostics
cfg = DiagnosticsConfig(experiment_dirs=EXPERIMENT_RESULT_DIRS, output_dir=DIAGNOSTICS_OUTPUT_DIR, models=MODELS,
                        chunk_size=CHUNK_SIZE, use_cache=USE_CACHE, split_seed=SPLIT_SEED,
                        hash_checkpoints=HASH_CHECKPOINTS,
                        run_frozen_inference_if_missing=RUN_FROZEN_INFERENCE_IF_MISSING)
print("experiment folders:", EXPERIMENT_RESULT_DIRS)
print("diagnostics output:", DIAGNOSTICS_OUTPUT_DIR)


In [ ]:
# --- 1. Which experiments/models were found? (searched only inside the configured folders) ---
import pandas as pd
from IPython.display import Image, Markdown, display
from decompositionality.diagnosis.artifacts import inventory
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
inv = inventory(cfg.experiment_dirs, pathlib.Path(cfg.output_dir) / "regenerated_predictions", cfg.models)
display(inv[["experiment", "model", "status", "original_predictions", "best_checkpoint", "manifest"]])
missing = inv[~inv.status.str.startswith("predictions available")]
if len(missing):
    print(f"{len(missing)} experiment/model combinations have NO predictions; they are reported as missing, never as 0.")
    if not RUN_FROZEN_INFERENCE_IF_MISSING:
        print("Set RUN_FROZEN_INFERENCE_IF_MISSING=True to regenerate them from the selected checkpoints (no training).")


## Run
One call performs the audit, catalog construction, matching, rule fitting, controls, figures and report. Re-running skips runs whose prediction-file hash and analysis version are unchanged.

In [ ]:
# --- 2. Run the full diagnostic pipeline (cached; CPU is sufficient; no model weights are loaded) ---
import time
_t = time.time()
R = run_diagnostics(cfg)
print(f"done in {time.time() - _t:.0f}s")
FIG = pathlib.Path(cfg.output_dir) / "figures"
def show(name):
    for p in sorted(FIG.glob(name + "*.png")):
        display(Image(filename=str(p)))


## Audit
Integer counts come from raw per-example records (not rounded summaries). `attribution_blocked` runs are excluded from all explanatory analyses.

In [ ]:
# --- 3. Artifact audit and integer success counts recomputed from raw records ---
cols = ["experiment", "model", "source", "n_records", "n_strings", "missing_strings", "duplicate_keys", "selected_step",
        "decoding", "invalid_records", "out_of_range_prediction_values", "target_oracle_mismatches",
        "input_manifest_mismatches", "stored_exact_mismatch", "attribution_blocked"]
display(R["audit"][[c for c in cols if c in R["audit"]]])
print("Test manifests identical across experiments:", R["manifests_identical_across_experiments"])
display(R["alignment"])
g = R["metrics_groups"]
display(g[["experiment", "model", "group", "exact_correct", "n", "exact_acc", "cell_correct", "cells_total", "cell_acc",
           "invalid_records", "invalid_rate"]].style.format({"exact_acc": "{:.6f}", "cell_acc": "{:.6f}", "invalid_rate": "{:.6f}"}, na_rep="N/A"))


In [ ]:
# --- 4. The 288-element transformation catalog and the 39 evaluated strings ---
print(R["catalog_check"])
display(R["string_classes"][R["string_classes"].experiment == "distinct_pairs"]
        .drop(columns=["experiment"]).style.hide(axis="index"))
print("Strings with the same group_id are aliases (same transformation). 'transformation_in_training' tracks the "
      "transformation, 'string_in_training' the symbolic string.")


## What the models output
Categories: *correct*; *wrong: unique legal* (exactly one of the 288 transformations reproduces the prediction); *wrong: ambiguous legal* (several do, e.g. symmetric inputs); *no legal match*. A match does not identify which symbolic string was executed.

In [ ]:
# --- 5. What do predictions match? (exact legal-transform matching; true accuracy shown first) ---
show("01_true_exact"); show("02_any_exact"); show("03_explanation"); show("04_dominant")
s = R["summary"]
display(s.style.format(precision=4, na_rep="N/A"))
top = R["confusion"]
top = top[top.bin == "wrong_unique_legal"].sort_values("count", ascending=False)
display(top.groupby(["experiment", "model"]).head(5)[["experiment", "model", "string", "word", "is_training_transform", "count", "n"]])


In [ ]:
# --- 6. Pre-declared human-readable error hypotheses (overlapping; need not sum to 1) ---
show("05_hypothesis")
h = R["hypotheses"]
display(h[(~h.truth_equivalent) & (h.compatible_rate > 0.2)].sort_values("compatible_rate", ascending=False)
        [["experiment", "model", "string", "hypothesis_type", "executed_word", "group_word", "aliases", "compatible_rate"]].head(40))


## Is the behaviour a stable rule?
Rules are fitted to **model predictions** (never targets) on the FIT half of the base grids and scored on the disjoint DIAG half. Many-to-one source-copy maps and bijective permutations are kept distinct; tie-driven/low-fidelity maps are flagged rather than called learned rules.

In [ ]:
# --- 7. Stable rules and general source maps (fitted on FIT grids from MODEL predictions; scored on DIAG grids) ---
show("08_stable"); show("10_source_maps")
sr = R["stable_rules"]
display(sr[["experiment", "model", "string", "selected_word", "n_tied_rules", "selected_is_truth", "selected_is_alt_generator",
            "selected_is_training_transform", "fit_exact_fidelity", "diag_exact_fidelity", "diag_cell_fidelity"]].head(60))
sm = R["source_maps"]
display(sm[sm.weak_or_tie_driven].groupby(["experiment", "model", "map_kind"]).size().rename("weak/tie-driven maps"))


## Composition and the half-period ambiguity
With Z = shift by (3, 3), the generators ZA, ZB, ZC reproduce every length-2 training example exactly but predict Z(target) for odd lengths. This checks an explicit explanation; it does not by itself show that this ambiguity caused the behaviour, and it cannot explain failures on unseen pairs.

In [ ]:
# --- 8. Composition consistency of fitted single-op maps, and the half-period (Z) ambiguity ---
show("09_composition"); show("07_alternate")
zt = R["z_table"]
display(zt.groupby(["experiment", "model", "parity"])[["exact_acc", "alt_generator_exact_rate", "alt_generator_cell_agreement"]]
        .mean().style.format("{:.4f}"))


In [ ]:
# --- 9. Calibration: what low cell accuracy can mean ---
show("06_nearest")
display(R["controls"]["summary"].style.format(precision=4))
display(R["controls"]["formula_check"])
print("A wrong permutation sharing k of 36 source positions with the truth has expected cell accuracy k/36 + (1-k/36)/10 "
      "on iid uniform digits; e.g. an extra right shift (k=0) gives 0.10 and exact agreement probability 10^-30, an extra Z "
      "gives 0.10 and 10^-18. Exact accuracy 0 with cell accuracy ~0.10 is therefore compatible with highly structured errors.")


In [ ]:
# --- 10. Paired comparison of the two experiments (same model, string and base grid) ---
show("11_paired")
p = R["paired"]
if len(p):
    display(p.groupby(["model"])[["identical_prediction_rate", "same_unique_wrong_element_rate",
                                  "delta_exact (all_pairs - distinct_pairs)", "delta_any_legal (all_pairs - distinct_pairs)",
                                  "delta_alt_generator_exact (all_pairs - distinct_pairs)"]].mean().style.format("{:.4f}"))
else:
    print("Paired comparison unavailable: a model needs predictions in BOTH experiments.")


In [ ]:
# --- 11. Fixed-ID qualitative examples (predeclared grid IDs, strings and models) ---
show("12_examples")


In [ ]:
# --- 12. Evidence-based report + compact export (cache excluded) ---
import zipfile
display(Markdown(pathlib.Path(R["report_path"]).read_text()))
out = pathlib.Path(cfg.output_dir)
arch = out.parent / (out.name + "_export.zip")
with zipfile.ZipFile(arch, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.rglob("*")):
        rel = f.relative_to(out)
        if f.is_file() and rel.parts[0] not in ("cache", "selfcheck", "regenerated_predictions"):
            z.write(f, pathlib.Path(out.name) / rel)
print("All derived outputs:", out)
print("Compact export:", arch, f"({arch.stat().st_size / 2**20:.1f} MiB)")


## Outputs
`<DIAGNOSTICS_OUTPUT_DIR>/`: `tables/` (CSV), `figures/` (PNG+PDF+CSV), `per_example/` (compact per-record diagnostics), `maps/` (all fitted source maps), `REPORT.md`, `provenance.json` (file hashes, selected steps, analysis code commit), `cache/`. Original predictions and checkpoints are only read.